In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = PROJECT_DIR / "OUTPUT"
OUTPUT_DIR.mkdir(exist_ok=True)

BOUNDARIES_DIR = RAW / "area_boundaries"

In [3]:
krs_file = list(BOUNDARIES_DIR.rglob("VG250_KRS.shp"))[0]
districts = gpd.read_file(krs_file)

districts = districts[districts["GF"] == 4][["AGS", "GEN", "geometry"]]
districts = districts.rename(columns={"AGS": "district_key"})

print(districts.shape)
print(districts.crs)
districts.head()

(400, 3)
EPSG:25832


,district_key,GEN,geometry
0,01001,Flensburg,"POLYGON ((526513.753 6075133.412, 526547.941 6..."
1,01002,Kiel,"POLYGON ((575841.569 6032148.032, 575869.668 6..."
2,01003,Lübeck,"POLYGON ((623056.151 5983746.445, 623191.574 5..."
3,01004,Neumünster,"POLYGON ((565015.652 6000637.513, 565128.417 6..."
4,01051,Dithmarschen,"MULTIPOLYGON (((479673.017 5990236.379, 479747..."


In [4]:
KEY_MAP = {"07235": "07211"}   # Trier-Saarburg -> Trier (same as in notebook 05)

districts["district_key"] = districts["district_key"].replace(KEY_MAP)
districts = districts.dissolve(by="district_key", as_index=False)
print("Districts after merge:", len(districts))

districts["geometry"] = districts.simplify(200)
districts = districts.to_crs(epsg=4326)

print(districts.crs)
districts.head(3)

Districts after merge: 399
EPSG:4326


,district_key,geometry,GEN
0,01001,"POLYGON ((9.41266 54.82264, 9.42293 54.82322, ...",Flensburg
1,01002,"POLYGON ((10.16916 54.43138, 10.18877 54.41108...",Kiel
2,01003,"POLYGON ((10.87684 53.98737, 10.88454 53.9782,...",Lübeck


In [5]:
districts["district_id"] = "DE" + districts["district_key"]
districts[["district_key", "district_id"]].head(3)

,district_key,district_id
0,01001,DE01001
1,01002,DE01002
2,01003,DE01003


In [6]:
panel = pd.read_csv(CLEAN_DIR / "panel_districts.csv", dtype={"district_key": str, "state_key": str})
base_name = panel["district_name"].str.split(",").str[0].str.strip()
is_city = (
    panel["district_name"].str.contains("kreisfreie Stadt|Stadtkreis")
    | panel["district_key"].isin(["02000", "11000"])
)
panel["district_label"] = base_name + is_city.map({True: " (city)", False: " (district)"})
panel.loc[panel["district_key"] == "07211", "district_label"] = "Trier + Trier-Saarburg (city + district)"

print("Unique labels:", panel[panel["year"] == 2026]["district_label"].nunique())

names = panel[panel["year"] == 2026][["district_key", "district_name", "district_label", "state", "region"]]

districts = districts[["district_key", "geometry"]].merge(names, on="district_key", how="left")
districts["district_id"] = "DE" + districts["district_key"]

print("Missing names:", districts["district_name"].isna().sum())

districts.to_file(OUTPUT_DIR / "districts.geojson", driver="GeoJSON")

size_mb = (OUTPUT_DIR / "districts.geojson").stat().st_size / 1_000_000
print(f"Saved: districts.geojson ({size_mb:.1f} MB)")

Unique labels: 399
Missing names: 0
Saved: districts.geojson (3.5 MB)


In [7]:
tableau = panel.copy()

labels = ["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"]
has_income = tableau["income_median_residence"].notna()
tableau.loc[has_income, "income_group"] = (
    tableau[has_income]
    .groupby("year")["income_median_residence"]
    .transform(lambda s: pd.qcut(s, 5, labels=labels))
    .astype(str)
)

outlier_keys = panel[(panel["year"] == 2026) & (panel["cars_per_1000_inh"] > 750)]["district_key"]
tableau["fleet_outlier"] = tableau["district_key"].isin(outlier_keys)

print(tableau.shape)
print(tableau["income_group"].value_counts(dropna=False))
print("Outlier districts:", tableau[tableau["year"] == 2026]["fleet_outlier"].sum())

(3990, 19)
income_group
NaN           1995
Q1 lowest      400
Q2             400
Q4             400
Q5 highest     400
Q3             395
Name: count, dtype: int64
Outlier districts: 6


In [8]:
tableau["district_id"] = "DE" + tableau["district_key"]
tableau[["district_key", "district_id"]].head(3)

,district_key,district_id
0,01001,DE01001
1,01001,DE01001
2,01001,DE01001


In [9]:
tableau.to_csv(OUTPUT_DIR / "tableau_panel.csv", index=False)
print("Saved: tableau_panel.csv | rows:", len(tableau))

Saved: tableau_panel.csv | rows: 3990
